Aflevering 3

zvp380/Josefine Rosenfeldt

8.oktober

In [10]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('cars.csv')
pd.set_option('display.float_format', '{:.2f}'.format)
# df.describe(percentiles=[]).T

In [ ]:
#4C - Monte Carlo

def simulate(n):

	##Parameterværdier
	beta0 = 2
	beta1 = 3
	beta2 = -1
	beta3 = -1

	# Step 2. Simuler data
	x1 = np.random.normal(loc=1, scale=2, size=n)   # x ~ N(1,4), dvs. std = 2
	x2 = np.random.normal(loc=1, scale=2, size=n)
	x3 = np.random.normal(loc=1, scale=2, size=n)
	eps = np.random.normal(loc=0, scale=0.5*x1**2)  # Var = 1/4 * x1^4, dvs. std = 0.5*x1^2
	y = beta0 + beta1*x1 + beta2*x2 + beta3*x3 + eps

	## Step 3: Estimer modellen med OLS
	X = pd.DataFrame({'x1': x1, 'x2': x2, 'x3': x3})
	X = sm.add_constant(X)
	results = sm.OLS(y, X).fit()

	beta1_hat = results.params['x1']    # Estimat af beta1
	se        = results.bse['x1']       # Almindelig standardfejl
	se_robust = results.HC1_se['x1']    # Robust standardfejl

	## Step 4: Test H0: beta1 = 3 (afvis hvis |t| > 1.96)
	rej        = abs((beta1_hat - 3) / se) > 1.96
	rej_robust = abs((beta1_hat - 3) / se_robust) > 1.96

	return beta1_hat, se, se_robust, rej, rej_robust


def monte_carlo(n, reps=1000):
	np.random.seed(0)
	out = []

	for rep in range(reps):
		out.append(simulate(n))

	results = pd.DataFrame(out, columns=['beta1', 'se', 'se_robust', 'rej', 'rej_robust'])

	return results


res = monte_carlo(n=50, reps=1000)
print(res.mean())
print(res['beta1'].std())